# Setup

## Import Packages and Functions

In [ ]:
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings("ignore")
import sys

from dataset import TrainDataset, TestDataset
from preprocessing import vjepa_preprocessing
from create_submission import encode_depth, save_submission
from fusion_transformer import DepthVJepaFusionTransformer

## Define Globals and Hyperparameters

In [ ]:
TRAIN_DIR = '../data/train'
TEST_DIR = '../data/test'
DEPTH_ANYTHING_MODEL = 'vitb' # vits, vitb, vitl, vitg
BATCH_SIZE = 8

## Load Train, Validation, and Test Data

In [ ]:
train_dataset = TrainDataset(TRAIN_DIR)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_fraction = 0.1
val_size = int(len(train_dataset) * val_fraction)
train_size = len(train_dataset) - val_size

train_subset, val_subset = random_split(train_dataset,
                                        [train_size, val_size],
                                        generator=torch.Generator())

train_loader_fit = DataLoader(train_subset,
                              batch_size=BATCH_SIZE,
                              shuffle=True,
                              num_workers=4,
                              pin_memory=True)
val_loader = DataLoader(val_subset,
                        batch_size=BATCH_SIZE,
                        num_workers=4,
                        pin_memory=True)

test_dataset = TestDataset(TEST_DIR)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

### Visualize Training Data

In [ ]:
batch = next(iter(train_loader))
images = batch["image"]
depths = batch["depth"]

print("Images:", images.shape)
print("Depths:", depths.shape)

_, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    img = images[i].permute(1, 2, 0) / 255.0
    depth = depths[i].squeeze(0)
    axes[0, i].imshow(img)
    axes[0, i].set_title("RGB image")
    axes[0, i].axis("off")
    im = axes[1, i].imshow(depth, cmap='viridis')
    axes[1, i].set_title('Depth map')
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

# V-JEPA Encoding

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
batch = next(iter(train_loader))

images = batch["image"].to(device)
print("original images:", images.shape)

images_vjepa = vjepa_preprocessing(images)
print("V-JEPA input:", images_vjepa.shape)

da_path = "../external/Depth-Anything-V2"
sys.path = [p for p in sys.path if p != da_path]
if "app" in sys.modules:
    del sys.modules["app"]

vj_encoder, _ = torch.hub.load('../external/vjepa2',
                            'vjepa2_1_vit_large_384',
                            source = 'local')

vj_encoder = vj_encoder.to(device).eval()

with torch.no_grad():
    tokens = vj_encoder(images_vjepa)

print("V-JEPA tokens:", tokens.shape)

# Depth Anything Depth Estimation

In [ ]:
sys.path.append("../external/Depth-Anything-V2")
from depth_anything_v2.dpt import DepthAnythingV2

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# see https://github.com/DepthAnything/Depth-Anything-V2#pre-trained-models
model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
    'vitg': {'encoder': 'vitg', 'features': 384, 'out_channels': [1536, 1536, 1536, 1536]}
}

da_encoder = DEPTH_ANYTHING_MODEL

da_model = DepthAnythingV2(**model_configs[da_encoder])
da_model.load_state_dict(torch.load(f'../checkpoints/depth_anything_v2_{da_encoder}.pth', map_location='cpu'))
da_model = da_model.to(device).eval()

## Show some Predictions of Train Data

In [ ]:
batch = next(iter(train_loader))
images = batch['image'].to(device)
print('input images:', images.shape)

images_da = images / 255.0

with torch.no_grad():
    pred_depth = da_model(images_da)

print('Depth Anything output:', pred_depth.shape)

images = batch["image"]
gt_depths = batch["depth"]
pred_depths = pred_depth

batch_size = BATCH_SIZE

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for i in range(4):
    rgb = images[i].detach().cpu().permute(1, 2, 0).numpy() / 255.0
    gt_depth = gt_depths[i, 0].detach().cpu().numpy()
    da_depth = pred_depths[i].detach().cpu().numpy()
    axes[0, i].imshow(rgb)
    axes[0, i].set_title(f"RGB {i}")
    axes[0, i].axis("off")
    axes[1, i].imshow(gt_depth, cmap='viridis')
    axes[1, i].set_title(f"GT depth {i}")
    axes[1, i].axis("off")
    axes[2, i].imshow(da_depth, cmap="viridis")
    axes[2, i].set_title(f"DA pred {i}")
    axes[2, i].axis("off")

plt.tight_layout()
plt.show()

# Training

## Define Loss Function

In [ ]:
def scale_invariant_rmse(pred, target, eps=1e-6):
    if pred.ndim == 4:
        pred = pred.squeeze(1)
    if target.ndim == 4:
        target = target.squeeze(1)

    mask = target > eps

    pred = torch.clamp(pred, min=eps)
    target = torch.clamp(target, min=eps)

    log_diff = torch.log(pred) - torch.log(target)
    log_diff = log_diff[mask]

    bias = - torch.mean(log_diff)
    return torch.sqrt(torch.mean((log_diff + bias)**2))

## Hyperparameters

In [ ]:
# AdamW Optimizer
LOSS_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# Training Loop
NUM_EPOCHS = 1
PATIENCE = 5

## Freeze DepthAnything and VJepa Models, Setup AdamW Optimizer

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
fusion_model = DepthVJepaFusionTransformer().to(device)

optimizer = torch.optim.AdamW(fusion_model.parameters(),
                              lr=LOSS_RATE,
                              weight_decay=WEIGHT_DECAY)

da_model.eval()
vj_encoder.eval()


## Main Training Loop

In [ ]:
best_val_loss = float('inf')
epochs_without_improvement = 0

for epoch in range(NUM_EPOCHS):
    fusion_model.train()

    total_loss = 0.0
    num_batches = 0

    for batch in train_loader_fit:
        images = batch["image"].to(device)
        targets = batch["depth"].to(device)

        optimizer.zero_grad()

        with torch.no_grad():
            da_depth = da_model(images_da / 255.0)
            vjepa_tokens = vj_encoder(vjepa_preprocessing(images))


        H, W = images.shape[-2:]

        refined_depth = fusion_model(
            da_depth=da_depth,
            vjepa_tokens=vjepa_tokens,
            output_size=(H, W),
        )

        loss = scale_invariant_rmse(refined_depth, targets)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1
        print(f"batch {num_batches}: loss = {loss.item():.6f}")

    mean_train_loss = total_loss / num_batches

    fusion_model.eval()
    val_loss = 0.0
    val_batches = 0

    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(device)
            targets = batch["depth"].to(device)

            da_depth = da_model(images / 255.0)
            vjepa_tokens = vj_encoder(vjepa_preprocessing(images))

            H, W = images.shape[-2:]

            refined_depth = fusion_model(da_depth=da_depth,
                                         vjepa_tokens=vjepa_tokens,
                                         output_size=(H,W))

            loss = scale_invariant_rmse(refined_depth, targets)

            val_loss += loss.item()
            val_batches += 1

        mean_val_loss = val_loss / val_batches

        print(
            f"Epoch {epoch + 1} | "
            f"train loss: {mean_train_loss:.6f} | "
            f"val loss: {mean_val_loss:.6f}"
        )

        if mean_val_loss < best_val_loss:
            best_val_loss = mean_val_loss
            epochs_without_improvement = 0

            torch.save(
            {
                "model_state_dict": fusion_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch,
                "val_loss": val_loss,
                "config": {
                    "vjepa_dim": 1024,
                    "d_model": 256,
                    "num_heads": 8,
                    "num_layers": 2,
                    "token_grid_size": 24,
                },
            },
            "../checkpoints/fusion_model_best.pth")

            print(f"Saved new best model with val loss {val_loss:.6f}")

        else:
            epochs_without_improvement += 1
            print(f"No improvement for {epochs_without_improvement}/{PATIENCE} epochs")

            if epochs_without_improvement >= PATIENCE:
                print("Early stopping")
                break



## Save Model

In [ ]:
torch.save(
    {
        "model_state_dict": fusion_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
        "loss": mean_loss,
        "config": {
            "vjepa_dim": 1024,
            "d_model": 256,
            "num_heads": 8,
            "num_layers": 2,
            "token_grid_size": 24,
        },
    },
    "../checkpoints/fusion_model_last.pth")

# Make Prediction

In [ ]:
# load model
checkpoint = torch.load("../checkpoints/fusion_model_last.pth", map_location=device)
fusion_model = DepthVJepaFusionTransformer(**checkpoint["config"]).to(device)
fusion_model.load_state_dict(checkpoint["model_state_dict"])
fusion_model.eval()

rows = []
with torch.no_grad():
    for batch in test_loader:
        images = batch["image"].to(device)
        image_ids = batch["id"]

        images_da = images / 255.0
        da_depth = da_model(images / 255.0)
        vjepa_tokens = vj_encoder(vjepa_preprocessing(images))
        H, W = images.shape[-2:]

        pred_depths = fusion_model(da_depth=da_depth,
                                         vjepa_tokens=vjepa_tokens,
                                         output_size=(H,W))
        if pred_depths.ndim == 4:
            pred_depths = pred_depths.squeeze(1)

        pred_depths = pred_depths.detach().cpu().numpy()

        for depth, image_id in zip(pred_depths, image_ids):
            rows.append({"id": f"{image_id}_depth","Depths": encode_depth(depth)})

df = save_submission(rows, "../submission.csv")